# Демо 1. Класичне донавчання: зміна голови в моделі BERT

У цьому ноутбуці донавчаємо модель **BERT** для задачі **визначення парафраз**: модель отримує два речення й визначає, чи передають вони той самий зміст.

План:
1. Завантажити датасет MRPC і токенізувати пари речень.
2. Завантажити BERT із **новою головою**-класифікатором.
3. **Повне донавчання** з `Trainer`: тренуються всі ваги (кістяк і голова).
4. Перевірити модель на власних прикладах.

Приклад за мотивами розділу 3 курсу [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter3/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [1]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [2]:
# Фіксуємо seed, щоб результати повторювалися між запусками
set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Пристрій: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
else:
    print("Включіть GPU, бо набридне чекати ;)")

Пристрій: cuda
GPU: Tesla T4


## 1. Дані: MRPC

**MRPC** (Microsoft Research Paraphrase Corpus) — пари речень з новинних статей.
Мітка `1` означає, що речення є парафразами, мітка `0` — що ні.

In [3]:
raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

mrpc/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  649kB            

mrpc/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 75.7kB            

mrpc/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  308kB            

mrpc/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [4]:
# Подивимося на один приклад
print("Приклад нееквівалентних речень:")
example = raw_datasets["train"][3]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

print("")
print("Приклад еквівалентних речень:")
example = raw_datasets["train"][5]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

Приклад нееквівалентних речень:
Речення 1: Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .
Речення 2: Tab shares jumped 20 cents , or 4.6 % , to set a record closing high at A $ 4.57 .
Мітка:    0 → not_equivalent

Приклад еквівалентних речень:
Речення 1: Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .
Речення 2: With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .
Мітка:    1 → equivalent


### Знайомимось з BERT BASE UNCASED

Для цього прочитайте інформацію на [картці моделі](https://huggingface.co/google-bert/bert-base-uncased)

In [5]:
from transformers import pipeline

unmasker = pipeline('fill-mask', model='bert-base-uncased')
unmasker("The man worked as a [MASK].")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[{'score': 0.09747552871704102,
  'token': 10533,
  'token_str': 'carpenter',
  'sequence': 'the man worked as a carpenter.'},
 {'score': 0.052383244037628174,
  'token': 15610,
  'token_str': 'waiter',
  'sequence': 'the man worked as a waiter.'},
 {'score': 0.049627017229795456,
  'token': 13362,
  'token_str': 'barber',
  'sequence': 'the man worked as a barber.'},
 {'score': 0.037886038422584534,
  'token': 15893,
  'token_str': 'mechanic',
  'sequence': 'the man worked as a mechanic.'},
 {'score': 0.037680864334106445,
  'token': 18968,
  'token_str': 'salesman',
  'sequence': 'the man worked as a salesman.'}]

In [6]:
print("Не по темі, але погляньте, яка \"несправедливість\"...")
unmasker("The woman worked as a [MASK].")

Не по темі, але погляньте, яка "несправедливість"...


[{'score': 0.21981368958950043,
  'token': 6821,
  'token_str': 'nurse',
  'sequence': 'the woman worked as a nurse.'},
 {'score': 0.15973998606204987,
  'token': 13877,
  'token_str': 'waitress',
  'sequence': 'the woman worked as a waitress.'},
 {'score': 0.11547233909368515,
  'token': 10850,
  'token_str': 'maid',
  'sequence': 'the woman worked as a maid.'},
 {'score': 0.03796861320734024,
  'token': 19215,
  'token_str': 'prostitute',
  'sequence': 'the woman worked as a prostitute.'},
 {'score': 0.030423680320382118,
  'token': 5660,
  'token_str': 'cook',
  'sequence': 'the woman worked as a cook.'}]

### Токенізація пари речень

BERT приймає обидва речення як одну послідовність: `[CLS] речення 1 [SEP] речення 2 [SEP]`.
Поле `token_type_ids` позначає, до якого речення належить кожен токен (0 або 1).

Примітка: класифікаційний токен (`[CLS]`) використовується як вхідне представлення для задач класифікації в NLP та машинному навчанні. Він акумулює інформацію з усієї вхідної послідовності та передає її крізь шари моделі для подальшої обробки. Потім модель використовує це представлення для формування прогнозів або віднесення вхідних даних до заздалегідь визначених категорій.

In [7]:
checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
encoded = tokenizer(example["sentence1"], example["sentence2"])

print("")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[:20])
print(encoded["token_type_ids"][:20])
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[-20:])
print(encoded["token_type_ids"][-20:])


['[CLS]', 'revenue', 'in', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[',', 'revenue', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.', '[SEP]']
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [8]:
def tokenize_function(batch):
    # truncation=True обрізає надто довгі пари до максимальної довжини моделі
    return tokenizer(batch["sentence1"], batch["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Доповнення (padding) робимо динамічно для кожного батча — до найдовшого прикладу в ньому
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_datasets

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

## 2. Модель: кістяк + нова голова

`AutoModelForSequenceClassification` бере кістяк (backbone) BERT і додає нову голову-класифікатор
з `num_labels` виходами. Зверніть увагу на звіт під час завантаження:
- ваги `cls.*` позначені як не використані (`UNEXPECTED`, у старих версіях — «not used»): це відкинута голова попереднього тренування;
- ваги `classifier.*` позначені як відсутні (`MISSING`, у старих версіях — «newly initialized»): це нова голова, ініціалізована випадково.

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

print(model.classifier)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Linear(in_features=768, out_features=2, bias=True)


In [10]:
def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if p.requires_grad or not trainable_only)


print(f"Кістяк (model.bert):   {count_params(model.bert):>12,} параметрів")
print(f"Голова (model.classifier): {count_params(model.classifier):>8,} параметрів")

Кістяк (model.bert):    109,482,240 параметрів
Голова (model.classifier):    1,538 параметрів


## 3. Повне донавчання з `Trainer`

`Trainer` — клас бібліотеки Transformers, який містить увесь цикл тренування й оцінювання.
Метрики рахуємо самі у функції `compute_metrics`.

Оновлюються **всі** ваги: і кістяк, і нова голова.
Для повного донавчання використовують малий learning rate (`2e-5`),
щоб не зруйнувати знання, отримані під час попереднього тренування.

In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }


training_args = TrainingArguments(
    output_dir="bert-mrpc",
    num_train_epochs=3,             # зазвичай вистачає 1-3 епохи
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",          # оцінюємо на валідації після кожної епохи
    save_strategy="no",             # чекпоінти не зберігаємо
    logging_steps=50,
    fp16=torch.cuda.is_available(), # змішана точність пришвидшує тренування на GPU
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [12]:
print(f"Параметрів, що тренуються: {count_params(model, trainable_only=True):,}")
trainer.train()
trainer.evaluate()

Параметрів, що тренуються: 109,483,778


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.487028,0.381455,0.838235,0.886986
2,0.328676,0.357605,0.848039,0.893471
3,0.167169,0.416541,0.860294,0.901554


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.167169,0.416541,3,0.860294,0.901554


{'eval_loss': 0.4165409803390503,
 'eval_accuracy': 0.8602941176470589,
 'eval_f1': 0.9015544041450777}

### **Питання 9**

Для перевірки впливу кількості епох на якість моделі було збільшено тривалість навчання з 3 до 6 епох, інші параметри без змін

In [15]:
# Завантажуємо модель заново для нового експерименту
set_seed(42)
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }


training_args = TrainingArguments(
    output_dir="bert-mrpc",
    num_train_epochs=6,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [17]:
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.480787,0.368141,0.838235,0.885813
2,0.328074,0.361860,0.833333,0.887043
3,0.150912,0.439773,0.879902,0.912656
4,0.096199,0.548645,0.867647,0.905594
5,0.032030,0.636767,0.867647,0.904594
6,0.037664,0.689757,0.855392,0.897391


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.037664,0.689757,6,0.855392,0.897391


{'eval_loss': 0.6897570490837097,
 'eval_accuracy': 0.8553921568627451,
 'eval_f1': 0.8973913043478261}

## 4. Перевірка на власних прикладах

In [13]:
def predict(model, sentence1, sentence2):
    model.eval()
    inputs = tokenizer(sentence1, sentence2, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = model(**inputs).logits.softmax(dim=-1)[0]
    return f"парафраз: {probs[1]:.1%}"


pairs = [
    ("The company reported a profit of $5 million.", "The firm earned five million dollars in profit."),
    ("The company reported a profit of $5 million.", "The weather was sunny all week."),
    ("She bought a new car yesterday.", "Yesterday she purchased a new vehicle."),
]
for s1, s2 in pairs:
    print(f"{s1}\n{s2}\n→ {predict(model, s1, s2)}\n")

The company reported a profit of $5 million.
The firm earned five million dollars in profit.
→ парафраз: 86.0%

The company reported a profit of $5 million.
The weather was sunny all week.
→ парафраз: 2.6%

She bought a new car yesterday.
Yesterday she purchased a new vehicle.
→ парафраз: 99.0%



### **Питання 10**

Додано власні 4 пари речень

In [18]:
# Чотири додаткові пари речень
additional_pairs = [
    # Однаковий зміст
    ("The meeting starts at nine in the morning.",
     "The meeting begins at 9 a.m."),

    ("Heavy rain caused the train to arrive late.",
     "The train arrived late because of heavy rain."),

    # Різний зміст
    ("The package was delivered on Monday.",
     "The package will be delivered next Monday."),

    ("The store is open on Sundays.",
     "The store is closed on Sundays."),
]

for s1, s2 in additional_pairs:
    print(f"{s1}\n{s2}\n→ {predict(model, s1, s2)}\n")

The meeting starts at nine in the morning.
The meeting begins at 9 a.m.
→ парафраз: 99.9%

Heavy rain caused the train to arrive late.
The train arrived late because of heavy rain.
→ парафраз: 99.9%

The package was delivered on Monday.
The package will be delivered next Monday.
→ парафраз: 99.9%

The store is open on Sundays.
The store is closed on Sundays.
→ парафраз: 99.9%



# **Відповіді на теоретичні питання**

## 1. Як довго виконувалися клітинки ноутбуку 01_bert_head? Чи були проблеми з запуском, які?

Виконання всіх клітинок ноутбука зайняло приблизно 2–3 хвилини. Найбільше часу витратилося на імпорт бібліотек (41 секунд) і донавчання моделі (близько 1 хвилини). Решта клітинок виконувалися переважно за кілька секунд.

Проблем із запуском не виникало, усі клітинки успішно виконалися на GPU Tesla T4. Однак під час роботи з'явилися певні попередження:

У комірці [3] під час завантаження датасету MRPC з'явилося попередження про відсутність токена Hugging Face (HF_TOKEN). Це означає, що завантаження відбувалося без авторизації, через що можуть діяти обмеження на кількість запитів. Але датасет успішно завантажився

У комірці [5] під час завантаження BERT з'явилося повідомлення UNEXPECTED. Воно означає, що частина ваг попередньо натренованої моделі не використовувалася, оскільки вони не потрібні для поточної задачі передбачення пропущених слів

У комірці [9] під час створення класифікатора з'явилися повідомлення UNEXPECTED і MISSING. Перше означає, що частина старих ваг не використовувалася, а друге, що нова класифікаційна голова отримала початкові випадкові ваги, які потрібно навчити

Ці повідомлення не є помилками, оскільки така поведінка очікувана під час використання BERT для різних задач. Вони не зупинили виконання програми: модель успішно завантажилася, пройшла донавчання та була перевірена на власних прикладах

## 2. Як Ви розумієте, що таке токен [CLS]? Яким чином він виконує свою функцію?

[CLS] - це спеціальний токен, який додається на початку вхідного тексту в моделі BERT. Він потрібен для того, щоб модель могла отримати загальне представлення всього тексту та використати його для класифікації.

Під час обробки тексту завдяки механізму уваги (self-attention) токен [CLS] взаємодіє з іншими токенами й збирає інформацію про їхній зміст та зв'язки між ними. На основі цієї інформації модель формує загальне представлення тексту, яке передається до класифікаційної голови для визначення результату.

У нашому випадку модель отримує два речення й визначає, чи є вони парафразами, тобто чи передають однаковий зміст, навіть якщо сформульовані різними словами.

## 3. Яке пряме застосування базової моделі bert-base-uncased?

Базова модель bert-base-uncased призначена насамперед для передбачення пропущених слів у тексті (Masked Language Modeling). Для цього частину тексту замінюють спеціальним токеном [MASK]. Модель аналізує слова до і після нього, враховує їхній зміст та зв'язки між ними, а потім визначає, які слова найкраще підходять на місце пропуску.

Наприклад, у нашому демо є речення «The man worked as a [MASK].» Модель аналізує контекст речення та пропонує найбільш імовірні слова, якими можна замінити [MASK].

Також під час попереднього навчання BERT використовувалося завдання Next Sentence Prediction, тобто чи є друге речення природним продовженням першого

Для виконання інших задач базову модель можна донавчити

## 4. Як Ви вважаєте, чи варто боротися (і якщо так, то яким чином) з проявами сексизму LLM-моделей?

Так, я вважаю, що з проявами сексизму в мовних моделях потрібно боротися, оскільки вони можуть відтворювати гендерні стереотипи, які містилися в даних для їхнього навчання.

Наприклад, у демо модель BERT частіше пов'язувала чоловіків із професіями столяра чи механіка, а жінок із професіями медсестри, офіціантки чи покоївки. Тобто модель застосовує певні стереотипні уявлення про професії

Щоб зменшити такі прояви, потрібно використовувати більш різноманітні та збалансовані навчальні дані, у яких професії та соціальні ролі не прив'язуються до певної статі. Також важливо перевіряти відповіді моделі на наявність стереотипів і за потреби додатково донавчати її на більш різноманітних прикладах. Це допоможе зробити відповіді моделей більш нейтральними та справедливими

## 5. Для чого в секції "Токенізація пари речень" виводяться числа 0 і 1?

Числа 0 і 1 використовуються для позначення того, до якого речення належить кожен токен. Модель BERT отримує два речення як одну послідовність. На початку послідовності ставиться спеціальний токен `[CLS]`, а токен `[SEP]` використовується для відокремлення першого речення від другого та позначення кінця другого речення

Для розрізнення речень використовується параметр `token_type_ids`. Число 0 позначає всі токени першого речення, включно з `[CLS]` і першим `[SEP]`. Число 1 позначає всі токени другого речення, включно з другим `[SEP]`

Таким чином, модель розуміє, які токени належать до кожного речення, що допомагає їй аналізувати зв'язок між ними та визначати, чи передають вони однаковий зміст

## 6. Що позначають числа 768 і 2, що виводяться після завантаження AutoModelForSequenceClassification.from_pretrained()?

Числа 768 і 2 позначають розміри входу та виходу останнього шару моделі BERT, який відповідає за класифікацію, тобто визначення класу тексту

768 - це кількість числових значень, якими BERT представляє зміст вхідного тексту. Останній шар моделі отримує ці значення з представлення токена [CLS] і використовує їх для визначення результату

2 - це кількість можливих класів, між якими модель має зробити вибір. У демо клас 0 означає, що два речення мають різний зміст, а клас 1 - що вони передають однаковий зміст

Отже, останній шар моделі отримує 768 значень і на їх основі обчислює дві оцінки, за якими модель визначає відповідний клас

## 7. Те що в кістяку (backbone) набагато більше параметрів ніж в голові, це добре чи погано в нашому випадку?

Це добре, оскільки кістяк (backbone) є основною частиною моделі BERT, яка вже навчена розуміти слова, їхній зміст і зв'язки між ними. Тому більшість параметрів зосереджена саме в ньому

Голова (head) - це невелика частина моделі, яка отримує інформацію від кістяка та визначає кінцевий результат. У демо вона має лише два можливі варіанти відповіді: речення мають однаковий зміст або різний

Така будова дозволяє використовувати вже отримані знання BERT для нової задачі. Але у демо донавчаються всі параметри, тому велика кількість параметрів у кістяку потребує більше пам'яті та часу на навчання

## 8. Що саме вимірюється за допомогою метрик accuracy & F1 в 01_bert_head?

Метрики accuracy та F1 використовуються для оцінювання того, наскільки правильно модель BERT визначає, чи мають два речення однаковий зміст

Accuracy показує частку всіх правильних відповідей моделі. Наприклад, якщо модель правильно визначила 80 зі 100 пар речень, її accuracy становить 80%

F1 показує, наскільки добре модель знаходить пари речень, які передають однаковий зміст. Вона поєднує два показники: precision (скільки пар, визначених як однакові за змістом, справді є такими) та recall (скільки з усіх пар з однаковим змістом модель змогла правильно знайти)

## 9. Збільшіть кількість епох тренування в 01_bert_head, покажіть (скопіювавши результати) що відбувається і чому.

Після збільшення кількості епох з 3 до 6 було помічено, що Training Loss загалом зменшувався, тобто модель краще засвоювала навчальні дані. Однак Validation Loss після другої епохи почав зростати та на шостій епосі досяг 0,689757

Найкращі показники Accuracy (87,99%) та F1 (91,27%) були отримані на третій епосі, а після шостої вони знизилися до 85,54% та 89,74%

Це свідчить про перенавчання (overfitting), коли модель починає надто сильно підлаштовуватися під навчальні дані, через що гірше працює з новими прикладами. Тому збільшення кількості епох не завжди покращує якість моделі

## 10. Додайте ще 4 перевірки на власних даних в пункті 4 в 01_bert_head.

Для додаткової перевірки моделі було створено чотири пари речень: дві з однаковим змістом і дві з різним

Модель для всіх чотирьох пар визначила ймовірність однакового змісту 99,9%. При цьому правильно було класифіковано лише дві пари з однаковим змістом. У двох інших випадках модель не врахувала важливі відмінності між реченнями, зокрема час події та протилежні значення слів

Тому модель не завжди правильно розрізняє зміст речень, навіть коли показує дуже високу впевненість у своїй відповіді. Це свідчить про те, що вона ще може помилятися на нових прикладах